## Part 1 - Data Preparation and Exploration 

In [1]:
%%capture
# Due to the configuration of the base Jupter image, the following imports are required for the regressions in the assignment to report the correct metrics

import sys 
!{sys.executable} -m pip uninstall statsmodels --yes 
!{sys.executable} -m pip uninstall numpy --yes
!{sys.executable} -m pip uninstall pandas --yes 
!{sys.executable} -m pip uninstall patsy --yes 
!{sys.executable} -m pip install numpy==1.17
!{sys.executable} -m pip install pandas==1.0
!{sys.executable} -m pip install patsy==0.5.2
!{sys.executable} -m pip install statsmodels==0.11.1

In [2]:
#Import Libraries

import pandas as pd
import datetime as dt
import scipy.stats as sp
import numpy as np
import statsmodels.formula.api as sm 

In [3]:
# Import Shotlog_14_15 and Player_Stats Datasets

Shotlog_1415=pd.read_csv("Assignment Data/Week 6/Shotlog_14_15.csv")
Player_Stats=pd.read_csv("Assignment Data/Week 6/Player_Stats_14_15.csv")
display(Shotlog_1415)

,game_id,date,match,home_team,away_team,home_away,result,final_margin,shot_number,quarter,...,closest_defender,closest_defender_id,closest_def_dist,current_shot_hit,points_earned,shoot_player,player_id,average_hit,shot_count,shot_per_game
0,21400280,5-Dec-14,ATL @ BKN,BKN,ATL,A,W,23,1,1,...,"Lopez, Brook",201572,6.6,1,2,al horford,201143,0.541259,715,10
1,21400280,5-Dec-14,ATL @ BKN,BKN,ATL,A,W,23,2,1,...,"Lopez, Brook",201572,5.6,0,0,al horford,201143,0.541259,715,10
2,21400280,5-Dec-14,ATL @ BKN,BKN,ATL,A,W,23,3,1,...,"Lopez, Brook",201572,4.7,0,0,al horford,201143,0.541259,715,10
3,21400280,5-Dec-14,ATL @ BKN,BKN,ATL,A,W,23,4,1,...,"Lopez, Brook",201572,5.8,0,0,al horford,201143,0.541259,715,10
4,21400280,5-Dec-14,ATL @ BKN,BKN,ATL,A,W,23,5,2,...,"Lopez, Brook",201572,6.4,0,0,al horford,201143,0.541259,715,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128064,21400350,14-Dec-14,WAS vs. UTA,WAS,UTA,H,W,9,11,3,...,"Burke, Trey",203504,4.7,1,2,john wall,202322,0.448513,874,15
128065,21400350,14-Dec-14,WAS vs. UTA,WAS,UTA,H,W,9,12,3,...,"Exum, Dante",203957,3.4,1,2,john wall,202322,0.448513,874,15
128066,21400350,14-Dec-14,WAS vs. UTA,WAS,UTA,H,W,9,13,4,...,"Kanter, Enes",202683,1.2,0,0,john wall,202322,0.448513,874,15
128067,21400350,14-Dec-14,WAS vs. UTA,WAS,UTA,H,W,9,14,4,...,"Kanter, Enes",202683,1.4,1,2,john wall,202322,0.448513,874,15


In [4]:
# Convert the “date” variable to a date type variable and calculate summary statistics for the “shot_clock” variable.

#Date variable conversion
Shotlog_1415['date']=pd.to_datetime(Shotlog_1415['date'])
Shotlog_1415['date'].describe()

count                  128069
unique                    120
top       2015-01-07 00:00:00
freq                     1941
first     2014-10-28 00:00:00
last      2015-03-04 00:00:00
Name: date, dtype: object

In [5]:
# calculate summary statistics for the “shot_clock” variable.
Shotlog_1415['shot_clock'].describe()

count    122502.000000
mean         12.453344
std           5.763265
min           0.000000
25%           8.200000
50%          12.300000
75%          16.675000
max          24.000000
Name: shot_clock, dtype: float64

In [6]:
Shotlog_1415.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128069 entries, 0 to 128068
Data columns (total 27 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   game_id               128069 non-null  int64         
 1   date                  128069 non-null  datetime64[ns]
 2   match                 128069 non-null  object        
 3   home_team             128069 non-null  object        
 4   away_team             128069 non-null  object        
 5   home_away             128069 non-null  object        
 6   result                128069 non-null  object        
 7   final_margin          128069 non-null  int64         
 8   shot_number           128069 non-null  int64         
 9   quarter               128069 non-null  int64         
 10  game_clock            128069 non-null  object        
 11  shot_clock            122502 non-null  float64       
 12  dribbles              128069 non-null  int64         
 13 

In [7]:
#Create a lagged variable “lag_shot_hit” to indicate the result of the previous shot by the same player at the same game.
Shotlog_1415['lag_shot_hit']=Shotlog_1415.sort_values(by=['player_id', 'game_id', 'shot_number'], ascending=[True, True, True]).groupby(['player_id', 'game_id'])['current_shot_hit'].shift(1)
Shotlog_1415.head()

,game_id,date,match,home_team,away_team,home_away,result,final_margin,shot_number,quarter,...,closest_defender_id,closest_def_dist,current_shot_hit,points_earned,shoot_player,player_id,average_hit,shot_count,shot_per_game,lag_shot_hit
0,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,1,1,...,201572,6.6,1,2,al horford,201143,0.541259,715,10,NaN
1,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,2,1,...,201572,5.6,0,0,al horford,201143,0.541259,715,10,1.0
2,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,3,1,...,201572,4.7,0,0,al horford,201143,0.541259,715,10,0.0
3,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,4,1,...,201572,5.8,0,0,al horford,201143,0.541259,715,10,0.0
4,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,5,2,...,201572,6.4,0,0,al horford,201143,0.541259,715,10,0.0


In [8]:
#Create a variable “error” to indicate the prediction error for each shot and a variable “lagerror” for the 
# prediction error for the previous shot. The “error” variable is defined as the difference between the outcome of the 
# current shot and the average success rate (“average_hit”) and the “lagerror” variable is defined as the difference 
# between the outcome of the previous shot and the average success rate.

Shotlog_1415['error']=Shotlog_1415['current_shot_hit']-Shotlog_1415['average_hit']
Shotlog_1415['lagerror']=Shotlog_1415['lag_shot_hit']-Shotlog_1415['average_hit']

In [9]:
# Calculate summary statistics for the “error” and “lagerror” variables. 
Shotlog_1415['error'].describe()

count    1.280690e+05
mean    -5.770049e-18
std      4.949640e-01
min     -7.124682e-01
25%     -4.491979e-01
50%     -3.850837e-01
75%      5.395973e-01
max      6.914894e-01
Name: error, dtype: float64

In [10]:
Shotlog_1415['lagerror'].describe()

count    113726.000000
mean          0.006303
std           0.496035
min          -0.712468
25%          -0.449198
50%          -0.382143
75%           0.542254
max           0.691489
Name: lagerror, dtype: float64

In [11]:
Player_Stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 281 entries, 0 to 280
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   shoot_player  281 non-null    object 
 1   average_hit   281 non-null    float64
dtypes: float64(1), object(1)
memory usage: 4.5+ KB


## Part 2 - Conditional Probability and Autocorrelation

In [12]:
# Create a dummy variable “conse_shot” that indicates a player made consecutive shots. 
Shotlog_1415['conse_shot_hit'] = np.where((Shotlog_1415['current_shot_hit']==1)&(Shotlog_1415['lag_shot_hit']==1), 1, 0)
Shotlog_1415.head()

,game_id,date,match,home_team,away_team,home_away,result,final_margin,shot_number,quarter,...,points_earned,shoot_player,player_id,average_hit,shot_count,shot_per_game,lag_shot_hit,error,lagerror,conse_shot_hit
0,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,1,1,...,2,al horford,201143,0.541259,715,10,NaN,0.458741,NaN,0
1,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,2,1,...,0,al horford,201143,0.541259,715,10,1.0,-0.541259,0.458741,0
2,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,3,1,...,0,al horford,201143,0.541259,715,10,0.0,-0.541259,-0.541259,0
3,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,4,1,...,0,al horford,201143,0.541259,715,10,0.0,-0.541259,-0.541259,0
4,21400280,2014-12-05,ATL @ BKN,BKN,ATL,A,W,23,5,2,...,0,al horford,201143,0.541259,715,10,0.0,-0.541259,-0.541259,0


In [13]:
Shotlog_1415 = Shotlog_1415.rename(columns={'conse_shot_hit': 'conse_shot'})

In [14]:
#Create a dataframe “Player_Prob” for the probability of making the previous shot and the joint probability for
# making both the previous and current shots. Name the probability of making the previous shot “average_lag_hit” and 
# the probability of making both shots “conse_shot_hit.” 

Player_Prob = Shotlog_1415.groupby(['shoot_player'])['conse_shot','lag_shot_hit'].mean()
Player_Prob=Player_Prob.reset_index()
Player_Prob.rename(columns={'lag_shot_hit':'average_lag_hit'}, inplace=True)
Player_Prob['conse_shot_hit']=Player_Prob['conse_shot']/Player_Prob['average_lag_hit']
Player_Prob.head()

,shoot_player,conse_shot,average_lag_hit,conse_shot_hit
0,aaron brooks,0.153298,0.418000,0.366741
1,aaron gordon,0.201923,0.532468,0.379221
2,al farouq aminu,0.162791,0.465686,0.349572
3,al horford,0.262937,0.537994,0.488736
4,al jefferson,0.207500,0.480000,0.432292


In [15]:
Player_Prob.rename(columns={'conse_shot':'conse_shot_hit','conse_shot_hit':'conditional_prob'}, inplace=True)

In [16]:
Player_Prob.head()

,shoot_player,conse_shot_hit,average_lag_hit,conditional_prob
0,aaron brooks,0.153298,0.418000,0.366741
1,aaron gordon,0.201923,0.532468,0.379221
2,al farouq aminu,0.162791,0.465686,0.349572
3,al horford,0.262937,0.537994,0.488736
4,al jefferson,0.207500,0.480000,0.432292


In [17]:
# Merge the “Player_Prob” dataframe into the “Player_Stats” dataframe.
Player_Stats=pd.merge(Player_Prob, Player_Stats, on=['shoot_player'])
Player_Stats.head(10)

,shoot_player,conse_shot_hit,average_lag_hit,conditional_prob,average_hit
0,aaron brooks,0.153298,0.418000,0.366741,0.415330
1,aaron gordon,0.201923,0.532468,0.379221,0.528846
2,al farouq aminu,0.162791,0.465686,0.349572,0.430233
3,al horford,0.262937,0.537994,0.488736,0.541259
4,al jefferson,0.207500,0.480000,0.432292,0.477500
5,alan anderson,0.157270,0.462366,0.340142,0.433234
6,alan crabbe,0.138298,0.523810,0.264023,0.425532
7,alex len,0.247492,0.539419,0.458811,0.528428
8,alexis ajinca,0.284360,0.598802,0.474882,0.597156
9,alonzo gee,0.137681,0.460784,0.298797,0.478261


In [18]:
Player_Stats.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 281 entries, 0 to 280
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   shoot_player      281 non-null    object 
 1   conse_shot_hit    281 non-null    float64
 2   average_lag_hit   281 non-null    float64
 3   conditional_prob  281 non-null    float64
 4   average_hit       281 non-null    float64
dtypes: float64(4), object(1)
memory usage: 13.2+ KB


In [19]:
# Calculate summary statistics for the probability for a player to make a shot (“average_hit”) and the 
# conditional probability for a player to make a shot given that he made the previous one (“conditional_prob”) and 
# the probability of players making consecutive shots (“conse_shot_hit”).

Player_Stats['average_hit'].describe()

count    281.000000
mean       0.451545
std        0.059392
min        0.308511
25%        0.413223
50%        0.446078
75%        0.480480
max        0.712468
Name: average_hit, dtype: float64

In [20]:
Player_Stats['conditional_prob'].describe()

count    281.000000
mean       0.380233
std        0.062320
min        0.225801
25%        0.336689
50%        0.381570
75%        0.422801
max        0.613209
Name: conditional_prob, dtype: float64

In [21]:
Player_Stats['conse_shot_hit'].describe()

count    281.000000
mean       0.176987
std        0.047943
min        0.076190
25%        0.144543
50%        0.171625
75%        0.203512
max        0.422392
Name: conse_shot_hit, dtype: float64

In [22]:
#Perform a t-test for the statistical significance on the difference between conditional probability 
#and unconditional probability of making a shot.

sp.stats.ttest_ind(Player_Stats['conditional_prob'], Player_Stats['average_hit'])

Ttest_indResult(statistic=-13.885932802814914, pvalue=6.925846314604593e-38)

In [23]:
#Calculate the first order autocorrelation coefficient on making a shot 
#(correlation coefficient between making the current shot and the previous shot) for the entire shotlog dataset.
Shotlog_1415['current_shot_hit'].corr(Shotlog_1415['lag_shot_hit'])

-0.010502388301693177

In [24]:
#Calculate the first order autocorrelation coefficient on making a shot for each player. 
#Display the top ten players with the highest first order autocorrelation coefficient.

Shotlog_1415.groupby('shoot_player')[['current_shot_hit','lag_shot_hit']].corr().head(10)

current_shot_hit  lag_shot_hit
shoot_player                                                    
aaron brooks    current_shot_hit          1.000000      0.002556
                lag_shot_hit              0.002556      1.000000
aaron gordon    current_shot_hit          1.000000     -0.071283
                lag_shot_hit             -0.071283      1.000000
al farouq aminu current_shot_hit          1.000000      0.029513
                lag_shot_hit              0.029513      1.000000
al horford      current_shot_hit          1.000000     -0.008396
                lag_shot_hit             -0.008396      1.000000
al jefferson    current_shot_hit          1.000000     -0.038885
                lag_shot_hit             -0.038885      1.000000

In [25]:
Autocorr_Hit=Shotlog_1415.groupby('shoot_player')[['current_shot_hit','lag_shot_hit']].corr().unstack().iloc[:,1].reset_index()
Autocorr_Hit.columns=Autocorr_Hit.columns.get_level_values(0)
Autocorr_Hit.rename(columns={'current_shot_hit':'autocorr'}, inplace=True)
Autocorr_Hit.head(10)

,shoot_player,autocorr
0,aaron brooks,0.002556
1,aaron gordon,-0.071283
2,al farouq aminu,0.029513
3,al horford,-0.008396
4,al jefferson,-0.038885
5,alan anderson,-0.022688
6,alan crabbe,0.028048
7,alex len,0.118461
8,alexis ajinca,0.047444
9,alonzo gee,-0.250290


In [26]:
Autocorr_Hit= Autocorr_Hit.sort_values(by='autocorr', ascending=False)
Autocorr_Hit.head(10)

,shoot_player,autocorr
131,joey dorsey,0.334252
54,cole aldrich,0.174666
200,nate robinson,0.122107
267,tyler hansbrough,0.120608
7,alex len,0.118461
50,cj mccollum,0.115949
114,jason smith,0.105903
190,matt bonner,0.098577
143,jusuf nurkic,0.097465
195,mike miller,0.089366


## Part 3 - Regression Analyses

In [27]:
# In this section, you will run several regressions to investigate the “hot hand.” 
#In all the regressions, the dependent variable is “error” and the independent variable of interest is “lagerror.” 

In [28]:
# Reg1: Run a linear least squares regression using the entire shotlog dataframe. Include the following control variables:

#Shot distance

#Number of dribbles

#Touch time

#Type of shot (“points” variable)

#Quarter of the game (as a categorical variable)

#Home or away game

#Shoot_player

#Closest defender

#Closest defender distance

In [29]:
Shotlog_1415.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128069 entries, 0 to 128068
Data columns (total 31 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   game_id               128069 non-null  int64         
 1   date                  128069 non-null  datetime64[ns]
 2   match                 128069 non-null  object        
 3   home_team             128069 non-null  object        
 4   away_team             128069 non-null  object        
 5   home_away             128069 non-null  object        
 6   result                128069 non-null  object        
 7   final_margin          128069 non-null  int64         
 8   shot_number           128069 non-null  int64         
 9   quarter               128069 non-null  int64         
 10  game_clock            128069 non-null  object        
 11  shot_clock            122502 non-null  float64       
 12  dribbles              128069 non-null  int64         
 13 

In [30]:
#Shotlog_1415['shoot_player'] = Shotlog_1415['shoot_player'].astype('category')
#Shotlog_1415['closest_defender'] = Shotlog_1415['closest_defender'].astype('category')

In [31]:
Shotlog_1415['error_demeaned'] = Shotlog_1415['error'] - Shotlog_1415.groupby('player_id')['error'].transform('mean')
Shotlog_1415['lagerror_demeaned'] = Shotlog_1415['lagerror'] - Shotlog_1415.groupby('player_id')['lagerror'].transform('mean')

In [32]:
Reg1 = sm.ols(formula = 'error_demeaned ~ lagerror_demeaned + shot_dist + dribbles + touch_time + C(points) + C(quarter) + home_away + closest_def_dist', data= Shotlog_1415).fit()
print(Reg1.summary())

                            OLS Regression Results                            
Dep. Variable:         error_demeaned   R-squared:                       0.041
Model:                            OLS   Adj. R-squared:                  0.041
Method:                 Least Squares   F-statistic:                     376.4
Date:                Mon, 13 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:51:53   Log-Likelihood:                -79034.
No. Observations:              113726   AIC:                         1.581e+05
Df Residuals:                  113712   BIC:                         1.582e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.1113      0.00

In [33]:
Reg2 = sm.wls(formula = 'error_demeaned ~ lagerror_demeaned + shot_dist + dribbles + touch_time + C(points) + C(quarter) + home_away + closest_def_dist',  weights=1/Shotlog_1415['shot_per_game'] , data= Shotlog_1415).fit()
print(Reg2.summary())

                            WLS Regression Results                            
Dep. Variable:         error_demeaned   R-squared:                       0.042
Model:                            WLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     380.7
Date:                Mon, 13 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:51:58   Log-Likelihood:                -88157.
No. Observations:              113726   AIC:                         1.763e+05
Df Residuals:                  113712   BIC:                         1.765e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.0975      0.00

In [34]:
#Reg3_player: Run linear least squares regressions on individual players. Include the following control variables:

#Shot distance

#Number of dribbles

#Touch time

#Type of shot (“points” variable)

#Quarter of the game (as a categorical variable)

#Home or away game

#Closest defender distance

# Reg4_wls_player: Run weighted least squares regressions on individual players. 
#Include the same set of control variables as in Reg3. 
#The regression should be weighted by the number of shot per game (weight=1/shot_per_game).

In [35]:
def Reg3_player(player):
    Shotlog_1415_player=Shotlog_1415[Shotlog_1415.shoot_player==player]
    Reg3_player=sm.ols(formula = 'error ~ lagerror + shot_dist + dribbles + touch_time + C(points) + C(quarter) + home_away + closest_def_dist', data= Shotlog_1415_player).fit()
    #print(Reg3_player.summary())
    return Reg3_player; 

In [36]:
def Reg4_wls_player(player):
    Shotlog_1415_player=Shotlog_1415[Shotlog_1415.shoot_player==player]
    Reg4_wls_player=sm.wls(formula = 'error ~ lagerror + shot_dist + dribbles + touch_time + C(points) + C(quarter) + home_away + closest_def_dist', weights=1/Shotlog_1415_player['shot_per_game'] , data= Shotlog_1415_player).fit()
    #print(Reg4_wls_player.summary())
    return Reg4_wls_player; 

In [37]:
player_list = np.array(Shotlog_1415['shoot_player'])
player_list = np.unique(player_list)

In [38]:
i = 0 
Player_Results_OLS = {}
while i <= len(player_list) - 1:
    #Shotlog_player=Shotlog[Shotlog.shoot_player==player_list[i]]
    reg_player = Reg3_player(player_list[i])
    RegParams = pd.DataFrame(reg_player.params).reset_index()
    RegTvals = pd.DataFrame(reg_player.tvalues).reset_index()
    RegPvals = pd.DataFrame(reg_player.pvalues).reset_index()

    RegOutput = pd.merge(RegParams, RegTvals, on=['index'])
    RegOutput = pd.merge(RegOutput, RegPvals, on=['index'])
    RegOutput
    
    LagErr = RegOutput[RegOutput['index'] == 'lagerror']
    LagErr = LagErr.drop(columns=['index'])
    LagErr = LagErr.rename(columns={"0_x":"Coef", "0_y":"T_Statistics", 0:"P_Value"})
    LagErr['shoot_player'] = player_list[i]
    Headers = ['shoot_player', 'Coef', 'T_Statistics', 'P_Value']
    Player_Results_OLS[i] = LagErr[Headers]
    i = i+1

In [39]:
RegPlayer_OLS = Player_Results_OLS[0]
j = 1
while j <= len(player_list) - 1:
    RegPlayer_OLS = RegPlayer_OLS.append(Player_Results_OLS[j])
    j = j+1
RegPlayer_OLS = RegPlayer_OLS.reset_index()
RegPlayer_OLS = RegPlayer_OLS.drop(columns=['index'])
RegPlayer_OLS

,shoot_player,Coef,T_Statistics,P_Value
0,aaron brooks,-0.004151,-0.093202,0.925781
1,aaron gordon,-0.079465,-0.656347,0.513918
2,al farouq aminu,0.096741,1.513574,0.131770
3,al horford,-0.002522,-0.067241,0.946411
4,al jefferson,-0.023667,-0.654268,0.513143
...,...,...,...,...
276,wesley matthews,-0.047961,-1.249311,0.211984
277,wilson chandler,-0.052219,-1.395720,0.163266
278,zach lavine,-0.016481,-0.298785,0.765306
279,zach randolph,0.049032,1.207406,0.227746


In [40]:
i = 0 
Player_Results_WLS = {}
while i <= len(player_list) - 1:
    #Shotlog_player=Shotlog[Shotlog.shoot_player==player_list[i]]
    reg_player = Reg4_wls_player(player_list[i])
    RegParams = pd.DataFrame(reg_player.params).reset_index()
    RegTvals = pd.DataFrame(reg_player.tvalues).reset_index()
    RegPvals = pd.DataFrame(reg_player.pvalues).reset_index()

    RegOutput = pd.merge(RegParams, RegTvals, on=['index'])
    RegOutput = pd.merge(RegOutput, RegPvals, on=['index'])
    RegOutput
    
    LagErr = RegOutput[RegOutput['index'] == 'lagerror']
    LagErr = LagErr.drop(columns=['index'])
    LagErr = LagErr.rename(columns={"0_x":"Coef", "0_y":"T_Statistics", 0:"P_Value"})
    LagErr['shoot_player'] = player_list[i]
    Headers = ['shoot_player', 'Coef', 'T_Statistics', 'P_Value']
    Player_Results_WLS[i] = LagErr[Headers]
    i = i+1

In [41]:
RegPlayer_WLS = Player_Results_WLS[0]
j = 1
while j <= len(player_list) - 1:
    RegPlayer_WLS = RegPlayer_WLS.append(Player_Results_WLS[j])
    j = j+1
RegPlayer_WLS = RegPlayer_WLS.reset_index()
RegPlayer_WLS = RegPlayer_WLS.drop(columns=['index'])
RegPlayer_WLS

,shoot_player,Coef,T_Statistics,P_Value
0,aaron brooks,-0.017039,-0.379413,0.704547
1,aaron gordon,-0.086933,-0.723936,0.471703
2,al farouq aminu,0.107612,1.706157,0.089587
3,al horford,0.005338,0.142661,0.886602
4,al jefferson,-0.042479,-1.176217,0.239888
...,...,...,...,...
276,wesley matthews,-0.024813,-0.643336,0.520225
277,wilson chandler,-0.039111,-1.054474,0.292050
278,zach lavine,-0.055391,-1.018136,0.309415
279,zach randolph,0.050400,1.246013,0.213240


In [44]:
pd.set_option('display.max_rows', 300)
display(RegPlayer_OLS)

,shoot_player,Coef,T_Statistics,P_Value
0,aaron brooks,-0.004151,-0.093202,0.925781
1,aaron gordon,-0.079465,-0.656347,0.513918
2,al farouq aminu,0.096741,1.513574,0.131770
3,al horford,-0.002522,-0.067241,0.946411
4,al jefferson,-0.023667,-0.654268,0.513143
5,alan anderson,0.007989,0.134741,0.892918
6,alan crabbe,0.071764,0.607910,0.545893
7,alex len,0.106514,1.736104,0.083885
8,alexis ajinca,0.011545,0.150916,0.880236
9,alonzo gee,-0.273124,-2.778511,0.006647


In [45]:
display(RegPlayer_WLS)

,shoot_player,Coef,T_Statistics,P_Value
0,aaron brooks,-0.017039,-0.379413,0.704547
1,aaron gordon,-0.086933,-0.723936,0.471703
2,al farouq aminu,0.107612,1.706157,0.089587
3,al horford,0.005338,0.142661,0.886602
4,al jefferson,-0.042479,-1.176217,0.239888
5,alan anderson,-0.023058,-0.388768,0.697757
6,alan crabbe,-0.001058,-0.008806,0.993008
7,alex len,0.115282,1.852375,0.065253
8,alexis ajinca,-0.025122,-0.310864,0.756317
9,alonzo gee,-0.353531,-3.607932,0.000507


In [46]:
A_Wiggins_reg=Reg3_player(player_list[17])
print(A_Wiggins_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  error   R-squared:                       0.078
Model:                            OLS   Adj. R-squared:                  0.064
Method:                 Least Squares   F-statistic:                     5.602
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           1.14e-08
Time:                        02:09:43   Log-Likelihood:                -500.04
No. Observations:                 738   AIC:                             1024.
Df Residuals:                     726   BIC:                             1079.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.2003      0.057  

In [47]:
J_Harden_reg=Reg3_player(player_list[108])
print(J_Harden_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  error   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     4.952
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           1.61e-07
Time:                        02:11:20   Log-Likelihood:                -691.15
No. Observations:                 995   AIC:                             1406.
Df Residuals:                     983   BIC:                             1465.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.1003      0.053  

In [48]:
S_Curry_reg=Reg3_player(player_list[247])
print(S_Curry_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  error   R-squared:                       0.074
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     6.498
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           1.79e-10
Time:                        02:12:54   Log-Likelihood:                -625.19
No. Observations:                 910   AIC:                             1274.
Df Residuals:                     898   BIC:                             1332.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.2196      0.057  

In [49]:
R_Westbrook_reg=Reg3_player(player_list[236])
print(R_Westbrook_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  error   R-squared:                       0.063
Model:                            OLS   Adj. R-squared:                  0.051
Method:                 Least Squares   F-statistic:                     5.529
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           1.32e-08
Time:                        02:14:48   Log-Likelihood:                -631.81
No. Observations:                 923   AIC:                             1288.
Df Residuals:                     911   BIC:                             1346.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.0958      0.051  